In [1]:
import random
import math
import numpy as np
import torch
from sklearn import datasets as sklearn_datasets
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import clear_output
import torch.nn as nn
import einops


import os, sys
sys.path.append("..")
sys.path.append("../ALAE")

In [24]:
#%cd ../ALAE

#! sed -i '/bimpy/d' requirements.txt
#! sed -i '/dareblopy/d' requirements.txt
#! sed -i 's/^sklearn$/scikit-learn/' requirements.txt

#! pip install -r requirements.txt
#! python training_artifacts/download_all.py
#%cd ../notebooks

In [4]:
class Sampler:
    def __init__(
        self, device='cuda',
    ):
        self.device = device

    def sample(self, size=5):
        pass


class TensorSampler(Sampler):
    def __init__(self, tensor, device='cuda'):
        super(TensorSampler, self).__init__(device)
        self.tensor = torch.clone(tensor).to(device)

    def sample(self, size=5):
        assert size <= self.tensor.shape[0]

        ind = torch.tensor(np.random.choice(np.arange(self.tensor.shape[0]), size=size, replace=False), device=self.device)
        return torch.clone(self.tensor[ind]).detach().to(self.device)


In [5]:
class EOTConfig:
    def __init__(self,
                 eps: float = 0.1,
                 batch_size: int = 2048,
                 device: str ="cpu",
                 K: int = 32,
                 epoch: int = 100,
                 lmc_steps: int = 100,
                 lmc_step_size: float = 0.003,
                 seed: int = 42,
                 grad_clip = 100000.0,
                 max_diff_exp_clip=100,
                 ema_momentum = 0.999
        ):
        self.device = device
        self.eps = eps
        self.batch_size = batch_size
        self.K = K
        self.epoch = epoch
        self.score_clip = 10000000.0
        self.grad_clip = grad_clip
        self.max_diff_exp_clip = max_diff_exp_clip
        self.lmc_steps = lmc_steps
        self.lmc_step_size = lmc_step_size
        self.seed = seed
        self.ema_momentum = ema_momentum

class MLP(nn.Module):
    def __init__(self, din=2, hidden=128, dout=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(din, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, dout),
        )

    def forward(self, x):
        return self.net(x)

In [6]:
DIM = 512
INPUT_DATA = "MAN"
TARGET_DATA = "WOMAN"

OUTPUT_SEED = 0xBADBEEF
EPSILON = 1.0

EXP_NAME = f'VarEOT_ALAE_{INPUT_DATA}_TO_{TARGET_DATA}_EPSILON_{EPSILON}'


## Data loading
### TO DOWNLOAD PRE-PROCESSED ALAE DATA, UNCOMMENT THE CODE OF THE NEXT CELL.


In [9]:
import gdown
import os

urls = {
    "../data/age.npy": "https://drive.google.com/uc?id=1Vi6NzxCsS23GBNq48E-97Z9UuIuNaxPJ",
    "../data/gender.npy": "https://drive.google.com/uc?id=1SEdsmQGL3mOok1CPTBEfc_O1750fGRtf",
    "../data/latents.npy": "https://drive.google.com/uc?id=1ENhiTRsHtSjIjoRu1xYprcpNd8M9aVu8",
    "../data/test_images.npy": "https://drive.google.com/uc?id=1SjBWWlPjq-dxX4kxzW-Zn3iUR3po8Z0i",
}

for name, url in urls.items():
    gdown.download(url, os.path.join(f"{name}"), quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1Vi6NzxCsS23GBNq48E-97Z9UuIuNaxPJ
To: /content/data/age.npy
100%|██████████| 560k/560k [00:00<00:00, 7.93MB/s]
Downloading...
From: https://drive.google.com/uc?id=1SEdsmQGL3mOok1CPTBEfc_O1750fGRtf
To: /content/data/gender.npy
100%|██████████| 1.68M/1.68M [00:00<00:00, 14.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1ENhiTRsHtSjIjoRu1xYprcpNd8M9aVu8
From (redirected): https://drive.google.com/uc?id=1ENhiTRsHtSjIjoRu1xYprcpNd8M9aVu8&confirm=t&uuid=037faf7d-4c42-424d-907e-b3fb70ff4a61
To: /content/data/latents.npy
100%|██████████| 143M/143M [00:03<00:00, 39.7MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1SjBWWlPjq-dxX4kxzW-Zn3iUR3po8Z0i
From (redirected): https://drive.google.com/uc?id=1SjBWWlPjq-dxX4kxzW-Zn3iUR3po8Z0i&confirm=t&uuid=e42adeb9-43e3-479e-9f73-04ba24a2a9bc
To: /content/data/test_images.npy
100%|██████████| 944M/944M [00:08<00:00, 107MB/s]


In [10]:
train_size = 60000
test_size = 10000

latents = np.load("../data/latents.npy")
gender = np.load("../data/gender.npy")
age = np.load("../data/age.npy")
test_inp_images = np.load("../data/test_images.npy")

train_latents, test_latents = latents[:train_size], latents[train_size:]
train_gender, test_gender = gender[:train_size], gender[train_size:]
train_age, test_age = age[:train_size], age[train_size:]

if INPUT_DATA == "MAN":
    x_inds_train = np.arange(train_size)[(train_gender == "male").reshape(-1)]
    x_inds_test = np.arange(test_size)[(test_gender == "male").reshape(-1)]
elif INPUT_DATA == "WOMAN":
    x_inds_train = np.arange(train_size)[(train_gender == "female").reshape(-1)]
    x_inds_test = np.arange(test_size)[(test_gender == "female").reshape(-1)]
elif INPUT_DATA == "ADULT":
    x_inds_train = np.arange(train_size)[
        (train_age >= 18).reshape(-1)*(train_age != -1).reshape(-1)
    ]
    x_inds_test = np.arange(test_size)[
        (test_age >= 18).reshape(-1)*(test_age != -1).reshape(-1)
    ]
elif INPUT_DATA == "CHILDREN":
    x_inds_train = np.arange(train_size)[
        (train_age < 18).reshape(-1)*(train_age != -1).reshape(-1)
    ]
    x_inds_test = np.arange(test_size)[
        (test_age < 18).reshape(-1)*(test_age != -1).reshape(-1)
    ]
x_data_train = train_latents[x_inds_train]
x_data_test = test_latents[x_inds_test]

if TARGET_DATA == "MAN":
    y_inds_train = np.arange(train_size)[(train_gender == "male").reshape(-1)]
    y_inds_test = np.arange(test_size)[(test_gender == "male").reshape(-1)]
elif TARGET_DATA == "WOMAN":
    y_inds_train = np.arange(train_size)[(train_gender == "female").reshape(-1)]
    y_inds_test = np.arange(test_size)[(test_gender == "female").reshape(-1)]
elif TARGET_DATA == "ADULT":
    y_inds_train = np.arange(train_size)[
        (train_age >= 18).reshape(-1)*(train_age != -1).reshape(-1)
    ]
    y_inds_test = np.arange(test_size)[
        (test_age >= 18).reshape(-1)*(test_age != -1).reshape(-1)
    ]
elif TARGET_DATA == "CHILDREN":
    y_inds_train = np.arange(train_size)[
        (train_age < 18).reshape(-1)*(train_age != -1).reshape(-1)
    ]
    y_inds_test = np.arange(test_size)[
        (test_age < 18).reshape(-1)*(test_age != -1).reshape(-1)
    ]
y_data_train = train_latents[y_inds_train]
y_data_test = test_latents[y_inds_test]

X_train = torch.tensor(x_data_train)
Y_train = torch.tensor(y_data_train)

X_test = torch.tensor(x_data_test)
Y_test = torch.tensor(y_data_test)

X_sampler = TensorSampler(X_train, device="cpu")
Y_sampler = TensorSampler(Y_train, device="cpu")

# Model initialisation


In [12]:
class EOTTrainer:
    def __init__(self, config, source_sampler, target_sampler, model_theta, model_phi, name):
        self.experiment_name = name

        self.config = config
        self.source_sampler = source_sampler
        self.target_sampler = target_sampler

        self.f_theta = model_theta
        self.f_phi = model_phi
        import copy
        self.f_theta_ema = copy.deepcopy(self.f_theta).eval()
        for p in self.f_theta_ema.parameters(): p.requires_grad_(False)
        self.current_step = 0

    def ema_update(self, model, ema):
        m = self.config.ema_momentum
        with torch.no_grad():
            for p, pe in zip(model.parameters(), ema.parameters()):
                pe.mul_(m).add_(p, alpha=1-m)


    def compute_loss(self, x, y):
        cfg = self.config
        broad_shape = list(x.shape)
        broad_shape.insert(1, cfg.K)
        z = torch.randn(size=broad_shape, device=cfg.device)

        x_noisy = x[:, None, :] - math.sqrt(cfg.eps(self.current_step)) * z

        fphi_x = self.f_phi(x)
        ftheta_xnoisy = self.f_theta(x_noisy.reshape(-1, *x_noisy.shape[2:])).view(cfg.batch_size, cfg.K) / cfg.eps(self.current_step)
        ftheta_y = self.f_theta(y)

        diff_in_exp = ftheta_xnoisy - fphi_x
        diff_in_exp_truncated = torch.clamp(diff_in_exp, min=None, max=self.config.max_diff_exp_clip)
        exp_term = torch.exp(diff_in_exp_truncated.to(torch.float64))

        ftheta_y_mean = ftheta_y.mean()
        fphi_x_mean = fphi_x.mean()
        Loss_theor = (fphi_x_mean + exp_term.mean()) * cfg.eps(self.current_step) - ftheta_y_mean
        L_main = Loss_theor
        return L_main

    def train_step(self):
        x = self.source_sampler.sample(self.config.batch_size).to(self.config.device)
        y = self.target_sampler.sample(self.config.batch_size).to(self.config.device)
        self.train_theta = True
        self.train_phi = True
        loss = self.compute_loss(x, y)
        self.opt_both.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.f_theta.parameters(), max_norm=self.config.grad_clip)
        torch.nn.utils.clip_grad_norm_(self.f_phi.parameters(), max_norm=self.config.grad_clip)
        self.opt_both.step()
        self.ema_update(self.f_theta, self.f_theta_ema)
        self.current_step += 1

    def train(self, viz_callback=None):
        print(f"Starting training name = {self.experiment_name}")
        print("-" * 60)

        while True:
            L_main = self.train_step()
            viz_callback(self)
            if (self.current_step >= self.config.epoch):
                break

        print("Training complete!")
        return 0


    def score_y_given_x(self, y, x):
        y = y.detach().requires_grad_(True)

        ft = self.f_theta_ema(y).sum()
        (gy,) = torch.autograd.grad(ft, y, retain_graph=False, create_graph=False)
        sc = (gy - (y - x)) / self.config.eps(self.current_step)

        return torch.clamp(sc, -self.config.score_clip, self.config.score_clip)

    def sample_pi_given_x(self, x, n=200):
        y = x.unsqueeze(1) + 0.0 * (self.config.eps(self.current_step) ** 0.5) * torch.randn(x.shape[0], n, x.shape[1], device=self.config.device)
        for _ in range(self.config.lmc_steps):
            y = y.detach()
            sc = self.score_y_given_x(y, x.unsqueeze(1))
            with torch.no_grad():
                y = y + self.config.lmc_step_size * sc + math.sqrt(2*self.config.lmc_step_size) * torch.randn_like(y)
        return y.detach()

    def sample_marginal(self, n_x=500):
        cfg = self.config
        xs = self.source_sampler.sample(n_x).to(cfg.device)
        y = xs

        for _ in range(cfg.lmc_steps):
            y = y.detach()
            sc = self.score_y_given_x(y, xs)

            with torch.no_grad():
                y = y + cfg.lmc_step_size * sc + \
                    math.sqrt(2 * cfg.lmc_step_size) * torch.randn_like(y)

        return y.detach()


    def save_checkpoint(self, filename):
        checkpoint = {
            'f_theta_state_dict': self.f_theta_ema.state_dict(),
            'f_phi_state_dict': self.f_phi.state_dict(),
            'epoch': self.current_step,
            'eps': self.config.eps(self.current_step),
        }
        torch.save(checkpoint, filename)

In [13]:
config = EOTConfig(
     eps = lambda step: EPSILON,
     batch_size = 256,
     device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
     K = 256,
     epoch = 5000,
     lmc_steps = 1000,
     lmc_step_size = 0.001,
     seed = 42,
     ema_momentum=0.999,
     max_diff_exp_clip=30,
     grad_clip=1e20
)

In [14]:
def visualize_training(trainer):
    if (trainer.current_step % 1000 == 0):
        print(trainer.current_step)

In [15]:
trainer = EOTTrainer(
    config=config,
    source_sampler=X_sampler,
    target_sampler=Y_sampler,
    model_theta=MLP(din=DIM, hidden=256).to(config.device),
    model_phi=MLP(din=DIM, hidden=256).to(config.device),
    name = f"alae_test"
)

trainer.opt_both = torch.optim.AdamW(
            list(trainer.f_phi.parameters()) + list(trainer.f_theta.parameters()),
            lr=3e-4,
            weight_decay=1e-4,
            betas = (0.7, 0.8)
        )

In [16]:
trainer.train(viz_callback=lambda t: visualize_training(t))

Starting training name = alae_test
------------------------------------------------------------
1000
2000
3000
4000
5000
Training complete!


0

# Results plotting


In [17]:
from tracker import RunningMeanTorch
torch.serialization.add_safe_globals([RunningMeanTorch])
from alae_ffhq_inference import load_model, encode, decode

model = load_model("../ALAE/configs/ffhq.yaml", training_artifacts_dir="../ALAE/training_artifacts/ffhq/").cuda()

In [20]:
N=7
repeat=3
torch.manual_seed(OUTPUT_SEED); np.random.seed(OUTPUT_SEED)
inds_to_map = np.random.choice(np.arange((x_inds_test < 300).sum()), size=N, replace=False)
mapped_all = []
latent_to_map = torch.tensor(test_latents[x_inds_test[inds_to_map]])
mapped = trainer.sample_pi_given_x(latent_to_map.to(trainer.config.device), n=repeat).to(trainer.config.device)

In [21]:
ref_plus_gen = torch.cat([latent_to_map.to(trainer.config.device).unsqueeze(1), mapped], dim=1)
with torch.no_grad():
    decoded_img = decode(model, ref_plus_gen.reshape(-1, 512))
    decoded_img = ((decoded_img * 0.5 + 0.5) * 255).type(torch.long).clamp(0, 255).cpu().type(torch.uint8).permute(0, 2, 3, 1).numpy()
    decoded_img = decoded_img.reshape(N, repeat + 1, 1024, 1024, 3)

In [22]:
def show_image_grid(images, figsize=(15, 6)):
    plt.close('all')
    %matplotlib inline
    if isinstance(images, torch.Tensor):
        images = images.cpu().numpy()

    grid = einops.rearrange(images, 'cols rows h w c -> (rows h) (cols w) c')

    if grid.max() > 1.0:
        grid = grid / 255.0

    plt.figure(figsize=figsize)
    plt.imshow(grid)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
show_image_grid(decoded_img)